In [26]:
import matplotlib.pyplot as plt
import pandas as pd
import ffmpeg
import os
import json
import plotly.express as px
import zipfile
import io
from hachoir.parser import createParser
from hachoir.metadata import extractMetadata

In [27]:
def get_videos(folder: str):
    files = []
    for file in os.listdir(folder):
        if file.endswith(".mp4"):
            files.append(file)
    return files

In [28]:
def get_total_duration(folder_path: str):
    total_seconds = 0.0
    folder = os.path.abspath(folder_path)
    
    for file in os.listdir(folder):
        # Catch both .mp4 and .avi
        if file.lower().endswith((".mp4", ".avi")):
            full_path = os.path.join(folder, file)
            
            try:
                probe = ffmpeg.probe(full_path)
                duration = float(probe['format']['duration'])
                total_seconds += duration
            except (ffmpeg.Error, KeyError, ValueError):
                print(f"Could not read duration for: {file}")
                continue

    total_hours = total_seconds / 3600
    
    hours = int(total_hours)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)
    
    return total_hours, f"{hours:02d}:{minutes:02d}:{seconds:02d}"

In [7]:
zip_path = '/run/media/sinal/SSD/videos.zip'
total_duration = 0.0 

with zipfile.ZipFile(zip_path, 'r') as z:
    for file_name in z.namelist():
        if file_name.endswith(('.mp4', '.mov')):
            with z.open(file_name) as f:
                # Read into RAM (be careful of file size vs RAM size!)
                file_stream = io.BytesIO(f.read())
                
                parser = createParser(file_stream)
                metadata = extractMetadata(parser)
                
                if metadata and metadata.has('duration'):
                    d = metadata.get('duration')
                    total_duration += d.total_seconds() 
                    
                file_stream.close()

hours = int(total_duration // 3600)
minutes = int((total_duration % 3600) // 60)
seconds = int(total_duration % 60)

print(f"Total Video Time: {hours}h {minutes}m {seconds}s")

[warn] Skip parser 'FAT12': Invalid FAT12 signature
[warn] Skip parser 'FAT16': Invalid FAT16 signature
[warn] Skip parser 'FAT32': Invalid FAT32 signature
[warn] Skip parser 'LinuxSwapFile': Unknown magic string
[warn] Skip parser 'MSDos_HardDrive': Invalid signature
[warn] Skip parser 'PIFVFile': Invalid magic number
[warn] Skip parser 'ElfFile': Invalid magic
[warn] Skip parser 'MachoFatFile': Invalid magic
[warn] Skip parser 'MachoFile': Invalid magic
[warn] Skip parser 'PRCFile': False
[warn] Skip parser 'FAT12': Invalid FAT12 signature
[warn] Skip parser 'FAT16': Invalid FAT16 signature
[warn] Skip parser 'FAT32': Invalid FAT32 signature
[warn] Skip parser 'LinuxSwapFile': Unknown magic string
[warn] Skip parser 'MSDos_HardDrive': Invalid signature
[warn] Skip parser 'PIFVFile': Invalid magic number
[warn] Skip parser 'ElfFile': Invalid magic
[warn] Skip parser 'MachoFatFile': Invalid magic
[warn] Skip parser 'MachoFile': Invalid magic
[warn] Skip parser 'PRCFile': False
[warn] S

Total Video Time: 18h 20m 20s


In [29]:
jig_hours = get_total_hours("/run/media/sinal/SSD/Suturing/video/")  + get_total_hours("/run/media/sinal/SSD/Needle_Passing/video/") + get_total_hours("/run/media/sinal/SSD/Knot_Tying/Knot_Tying/video/")
print(jig_hours)

5.268957698333333


In [41]:
data = {
    'Dataset': ['PitVis', 'CATARACTS', 'AUTOLAPARO', 'JIGSAWS', 'Cholec80'],
    'Frames': [int((get_total_hours("/home/sinal/Downloads/PitVis/") * 3600) * 24), (9 * 3600) * 30 + (9 * 3600) * 50, (23 * 3600) * 25, (int(jig_hours) * 3600) * 30, ((102 * 3600) * 25) / 2],
    'Type': ['Pretraining', 'Pretraining', 'Pretraining', 'Pretraining', 'Evaluation']
}



df = pd.DataFrame(data)

In [42]:
# Sort the dataframe by hours
df = df.sort_values(by='Frames', ascending=False)

# Bar chart
fig = px.bar(
    df, 
    x='Dataset', 
    y='Frames', 
    color='Type',
    text='Frames',
    title='Dataset Sizes: Pretraining vs Evaluation',
    labels={'Frames': 'Total Frames', 'Dataset': 'Dataset Name'},
    color_discrete_map={'Evaluation': '#EF553B', 'Pretraining': '#636EFA'},
    template='plotly_white'
)

fig.update_traces(
    texttemplate='%{text:,}', 
    textposition='outside',
    marker_line_color='rgb(8,48,107)',
    marker_line_width=1.5,
    opacity=0.8
)

fig.update_layout(
    xaxis_tickangle=-45,
    uniformtext_minsize=8,
    uniformtext_mode='hide',
    showlegend=True,
    legend_title_text='Usage Type',
    font=dict(size=14)
)

fig.show(renderer='browser')
fig.write_html("dataset_plotly.html")